# Fig S2 (E13.5) — Number of Genes Retained for CellChat Signaling Communication Inference Analysis Using Different Trim Parameters

This notebook quantifies, for each Oi-cluster, how many genes are detected as "over-expressed" in CellChat analysis and retained for subsequent signaling communication inference after filtering genes by different trimming thresholds (`trim` = 0.1, 0.05, 0.01).

**Workflow**
1. Load the E13.5 bin50 AnnData object and the CellChat over-expressed gene list
   (exported to CSV from the `@var.features` slot of the CellChat object after
   running `identifyOverExpressedGenes()`).
2. For each trim value, identify detected genes per Oi-cluster — defined as
   genes whose fraction of non-zero expression spots within the cluster is strictly greater
   than the trim threshold — and intersect them with the over-expressed gene set to get the retained genes.
3. Plot the per-Oi-cluster overlap counts across trim values.

## 1. Setup

In [ ]:
import os

import numpy as np
import pandas as pd
import anndata as ad
import seaborn as sns
import matplotlib.pyplot as plt

# Use Arial as the default figure font
plt.rcParams["font.family"] = "Arial"

## 2. Load Data

In [ ]:
# Load bin50 AnnData (E13.5_E1S1)
bin50_data = ad.read_h5ad("/home/project_interconnectivity/Mouse_spatialtranscriptome_mosta_bgi_2022/Seurat_integration_E9.5+E13.5/E9.5_bin50+E13.5_bin50/SCTransform+CCA_v2/res=0.7/E13.5_E1S1_bin50.h5ad")
bin50_data = bin50_data.copy()

seurat_clusters = bin50_data.obs['seurat_clusters'].unique()
bin50_data

In [ ]:
# Load CellChat over-expressed gene list
# exported to CSV from the `@var.features` slot of the CellChat object after running `identifyOverExpressedGenes()`
overexpressed_genes_df = pd.read_csv("/home/project_interconnectivity/Mouse_spatialtranscriptome_mosta_bgi_2022/Seurat_integration_E9.5+E13.5/E9.5_bin50+E13.5_bin50/SCTransform+CCA_v2/res=0.7/E13.5_overExpr_genes_list.csv")
overexpressed_genes = overexpressed_genes_df['x'].unique()

print(f"Number of over-expressed genes: {len(overexpressed_genes)}")

## 3. Detect Genes Retained per Cluster for Each Trim Value

In [ ]:
def get_detected_overExpr_genes_info(data, fraction, overexpressed_genes):
    """For each cluster, find genes whose non-zero expression fraction exceeds `fraction` (trim threshold),
    then count how many of those overlap with the over-expressed gene list.

    Parameters:
    data : AnnData
        Expression data with `seurat_clusters` (where Oi-cluster identities stored in) in `.obs`.
    fraction : float
        Trim threshold; a gene is retained in an Oi-cluster if its fraction of
        non-zero spots is strictly greater than this value.
    overexpressed_genes : array-like
        Reference set of over-expressed gene IDs.
    ----------

    Returns:
    pd.DataFrame
        One row per Oi-cluster with retained-gene IDs/counts and overlap with the
        over-expressed gene set.
    ----------
    """
    overexpressed_set = set(overexpressed_genes)
    cluster_detected_genes = []

    for cluster in seurat_clusters:
        cluster_data = data[data.obs['seurat_clusters'] == cluster].copy()
        data_X = cluster_data.X.toarray() if hasattr(cluster_data.X, 'toarray') else cluster_data.X

        detected_gene_IDs = []
        for i, gene in enumerate(cluster_data.var_names):
            gene_expr = data_X[:, i].flatten()
            total_count = len(gene_expr)
            nonzero_perc = np.count_nonzero(gene_expr) / total_count if total_count > 0 else 0
            if nonzero_perc > fraction:
                detected_gene_IDs.append(gene)

        overlap_genes = list(set(detected_gene_IDs) & overexpressed_set)

        cluster_detected_genes.append({
            'Oi_cluster': cluster,
            'trim': fraction,
            'detected_gene_IDs': detected_gene_IDs,
            'detected_gene_counts': len(detected_gene_IDs),
            'retained_gene_IDs': overlap_genes,
            'retained_gene_count': len(overlap_genes),
        })

    return pd.DataFrame(cluster_detected_genes)

In [ ]:
# Compute detected/overlap info for each trim value
trim_values = [0.1, 0.05, 0.01]

detected_dfs = {
    trim: get_detected_overExpr_genes_info(bin50_data, trim, overexpressed_genes)
    for trim in trim_values
}

for trim, df in detected_dfs.items():
    print(f"--- trim = {trim} ---")
    print(df)
    print()

In [ ]:
# Combine all trim results into a single tidy DataFrame
aggregated_detected_overExpr_genes_df = pd.concat(
    detected_dfs.values(), ignore_index=True
)
aggregated_detected_overExpr_genes_df

## 4. Plot — Detected Over-Expressed Genes per Cluster

In [ ]:
plot_df = aggregated_detected_overExpr_genes_df.copy()
plot_df['Oi_cluster'] = plot_df['Oi_cluster'].astype(int)
plot_df = plot_df.sort_values(by='Oi_cluster')

sns.set_theme(style="whitegrid")
plt.figure(figsize=(12, 6))

sns.scatterplot(
    data=plot_df,
    x='Oi_cluster',
    y='retained_gene_count',
    hue='trim',
    palette=sns.color_palette("deep6", len(trim_values)),
    s=80,
)

plt.title(
    "Detected Over-Expressed Gene Numbers after Trimming per Oi-cluster | "
    "Oi-cluster| res=0.7 | bin50 | E13.5"
)
plt.xlabel("Oi-cluster")
plt.ylabel("Number of retained genes")
plt.xticks(ticks=range(1, 32), labels=range(1, 32))
plt.legend(
    title="Trim Value",
    bbox_to_anchor=(1.02, 1),
    loc='upper left',
    borderaxespad=0,
)
plt.tight_layout()
plt.show()